In [ ]:
#@title Run
Mount_Google_Drive = True #@param {type:"boolean"}

if Mount_Google_Drive:
    from google.colab import drive
    print("📂 Connecting to Google Drive...")
    drive.mount('/content/drive')

import os, sys, time, base64, subprocess, shutil

ENCODED_REPO = "aHR0cHM6Ly9naXRodWIuY29tL3lhcmFuYmFyemkveWFyYWJhcnppLWFpZ29sZGVuLWR1Yi5naXQ="
REPO_URL = base64.b64decode(ENCODED_REPO.encode("utf-8")).decode("utf-8")
APP_DIR = "/content/dubbing-app"

subprocess.run("fuser -k 3000/tcp 2>/dev/null || true", shell=True)
subprocess.run("pkill -9 -f cloudflared 2>/dev/null || true", shell=True)

def is_node20_installed():
    try:
        res = subprocess.run(["node","-v"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return res.returncode == 0 and res.stdout.strip().startswith("v20.")
    except:
        return False

def run_cmd(cmd, cwd=None):
    res = subprocess.run(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if res.stdout.strip():
        print(res.stdout.strip())
    return res.returncode

if not is_node20_installed():
    print("🔧 Setting up Node.js 20 & FFmpeg...")
    subprocess.run("apt-get purge -y nodejs npm libnode-dev gyp 2>/dev/null || true", shell=True)
    subprocess.run("apt-get autoremove -y 2>/dev/null || true", shell=True)
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1", shell=True)
    subprocess.run("apt-get install -y nodejs ffmpeg build-essential >/dev/null 2>&1", shell=True)
    time.sleep(3)
else:
    print("✅ Node.js 20 already present. Ensuring FFmpeg...")
    subprocess.run("apt-get install -y ffmpeg >/dev/null 2>&1", shell=True)

print("📥 Preparing backend app...")
if os.path.exists(APP_DIR):
    shutil.rmtree(APP_DIR)
run_cmd(f"git clone --depth 1 {REPO_URL} {APP_DIR}")
run_cmd("npm install", cwd=APP_DIR)

print("🌐 Creating public tunnel...")
subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared", shell=True)
subprocess.run("chmod +x /usr/local/bin/cloudflared", shell=True)

node_proc = subprocess.Popen(["npm","run","dev"], cwd=APP_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
cf_proc = subprocess.Popen(["cloudflared","tunnel","--url","http://localhost:3000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

app_url = None
start_time = time.time()
while time.time() - start_time < 90:
    line = cf_proc.stdout.readline()
    if line:
        print(line.strip())
        if "https://" in line and ".trycloudflare.com" in line:
            import re
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                app_url = match.group(0)
                break
    if node_proc.poll() is not None:
        print("❌ Node server stopped. Re-run this cell.")
        break
    time.sleep(0.3)

if app_url:
    with open("/content/app_url.txt", "w") as f:
        f.write(app_url)
    print("\n" + "="*55)
    print("✅ BACKEND READY")
    print("🔗", app_url)
    print("➡️  Keep this cell running. Then run the UI cell below.")
    print("="*55)
else:
    print("❌ Tunnel could not start. Re-run this cell.")


## ✅ حالت استفاده
1. سلول **Run** را اجرا کن و صبر کن `BACKEND READY` بیاید.
2. سلول دوم را اجرا کن.
3. **Upload media** را بزن و فایل را انتخاب کن.
4. فقط ۱ → ۲ → ۳ را به ترتیب بزن، یا **Run all 3 steps**. اگر فایل بلندتر از ۵ دقیقه باشد خودش تقسیم و آخر به هم می‌چسباند.


In [ ]:
#@title 🎬 AI Studio Dubbing UI (auto 5-min chunks)
#@markdown ### ⚙️ Settings
voice_name = "charon" #@param ["charon", "puck", "kore", "fenrir", "aoede"]
source_language = "English" #@param ["English", "Auto-Detect"]
target_language = "Persian" #@param ["Persian", "English", "Spanish", "French", "German", "Chinese", "Japanese", "Arabic", "Russian"]
#@markdown اگر فایل کمتر از ۵ دقیقه باشد مستقیم پردازش می‌شود؛ اگر بلندتر باشد خودکار به تکه‌های ۵ دقیقه‌ای تقسیم و آخر به هم چسبانده می‌شود.

import os, shutil, subprocess, time, re
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import files
from gradio_client import Client, handle_file

CHUNK_SEC = 300
if not os.path.exists("/content/app_url.txt"):
    raise RuntimeError("اولا سلول Run بالای همین نوت‌بوک را اجرا کن تا BACKEND READY بشود.")
SPACE = open("/content/app_url.txt").read().strip()
print("Using backend:", SPACE)

os.makedirs("input_media", exist_ok=True)
os.makedirs("output_media", exist_ok=True)
state = {"input": ""}

upload_btn = widgets.Button(description="📤 Upload media", button_style="primary", icon="upload")
btn_transcribe = widgets.Button(description="1. Transcribe", button_style="info", icon="microphone")
btn_translate = widgets.Button(description="2. Translate", button_style="warning", icon="language")
btn_dub = widgets.Button(description="3. Dub", button_style="success", icon="play")
btn_all = widgets.Button(description="▶ Run all 3 steps", button_style="danger")
log = widgets.Output(layout={"border": "1px solid gray"})


def run(cmd):
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)


def media_duration(path):
    return float(subprocess.check_output([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=nw=1:nk=1", path
    ]).decode().strip())


def split_media(src, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    duration = media_duration(src)
    ext = os.path.splitext(src)[1] or ".mp4"
    parts, offsets = [], []
    start, i = 0.0, 0
    while start < duration - 0.05:
        part = os.path.join(out_dir, "part_%03d%s" % (i, ext))
        run(["ffmpeg", "-y", "-ss", ("%.3f" % start), "-t", str(CHUNK_SEC),
             "-i", src, "-c", "copy", "-avoid_negative_ts", "make_zero", part])
        parts.append(part)
        offsets.append(start)
        start += CHUNK_SEC
        i += 1
    return parts, offsets


def save_result(result, dest):
    candidates = result if isinstance(result, (list, tuple)) else [result]
    for item in candidates:
        if isinstance(item, str) and os.path.exists(item):
            shutil.copy(item, dest)
            return dest
    with open(dest, "w", encoding="utf-8") as f:
        f.write(str(result))
    return dest


def shift_srt(text, offset_ms):
    def stamp(m):
        h, mi, s, ms = map(int, m.groups())
        total = h * 3600000 + mi * 60000 + s * 1000 + ms + int(offset_ms)
        hh, rem = divmod(total, 3600000)
        mm, rem = divmod(rem, 60000)
        ss, mss = divmod(rem, 1000)
        return "%02d:%02d:%02d,%03d" % (hh, mm, ss, mss)
    return re.sub(r"(\d{2}):(\d{2}):(\d{2}),(\d{3})", stamp, text)


def join_srts(paths, offsets, dest):
    blocks, n = [], 1
    for path, off in zip(paths, offsets):
        text = shift_srt(open(path, encoding="utf-8", errors="ignore").read(), off * 1000)
        text = text.replace("\r\n", "\n")
        for block in [b for b in text.split("\n\n") if b.strip()]:
            lines = block.strip().splitlines()
            if len(lines) >= 2:
                lines[0] = str(n)
                blocks.append("\n".join(lines))
                n += 1
    with open(dest, "w", encoding="utf-8") as f:
        f.write("\n\n".join(blocks) + "\n")
    return dest


def concat_media(parts, dest):
    lst = dest + ".txt"
    with open(lst, "w", encoding="utf-8") as f:
        for p in parts:
            f.write("file '%s'\n" % os.path.abspath(p))
    run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", lst, "-c", "copy", dest])
    return dest


def call_api(api_name, media_path):
    client = Client(SPACE)
    return client.predict(
        media_file=handle_file(media_path),
        voice_name=voice_name,
        source_lang=source_language,
        target_lang=target_language,
        api_name=api_name,
    )


def process_auto(api_name, label):
    if not state["input"]:
        print("❌ اول Upload media را بزن.")
        return
    src = state["input"]
    duration = media_duration(src)

    if duration <= CHUNK_SEC:
        print("⏳ %s (direct, %.1f min)..." % (label, duration / 60.0))
        result = call_api(api_name, src)
        if api_name == "/dub_only":
            ext = os.path.splitext(result)[1] if isinstance(result, str) and os.path.exists(result) else os.path.splitext(src)[1]
            final = save_result(result, os.path.join("output_media", "dubbed" + ext))
        else:
            final = save_result(result, os.path.join("output_media", label + ".srt"))
        print("✅ Done: " + final)
        files.download(final)
        return

    work = "output_media/chunks_" + label
    if os.path.exists(work):
        shutil.rmtree(work)
    os.makedirs(os.path.join(work, "in"), exist_ok=True)

    print("✂️ Auto-splitting into %d-minute chunks..." % (CHUNK_SEC // 60))
    parts, offsets = split_media(src, os.path.join(work, "in"))
    print("✅ %d chunks created." % len(parts))

    done = []
    for i, part in enumerate(parts, 1):
        print("⏳ %s chunk %d/%d ..." % (label, i, len(parts)))
        err = None
        for attempt in range(1, 4):
            try:
                result = call_api(api_name, part)
                err = None
                break
            except Exception as e:
                err = e
                print("   attempt %d failed; waiting 20s: %s" % (attempt, e))
                time.sleep(20)
        if err:
            raise err

        if api_name == "/dub_only":
            ext = os.path.splitext(result)[1] if isinstance(result, str) and os.path.exists(result) else os.path.splitext(part)[1]
        else:
            ext = ".srt"
        out = os.path.join(work, "out_%03d%s" % (i, ext))
        save_result(result, out)
        done.append(out)
        print("✅ chunk %d done." % i)

    if api_name == "/dub_only":
        final = os.path.join("output_media", "dubbed_full" + os.path.splitext(done[0])[1])
        concat_media(done, final)
    else:
        final = os.path.join("output_media", label + "_full.srt")
        join_srts(done, offsets, final)

    print("📥 Final file: " + final)
    files.download(final)


def on_upload(_):
    with log:
        clear_output()
        state["input"] = ""
        print("📤 Please upload your Video or Audio file:")
        uploaded = files.upload()
        if not uploaded:
            print("❌ No file selected.")
            return
        filename = next(iter(uploaded))
        dest = os.path.join("input_media", filename)
        shutil.move(filename, dest)
        state["input"] = dest
        try:
            print("✅ Saved: %s  (%.1f min)" % (dest, media_duration(dest) / 60.0))
        except Exception:
            print("✅ Saved: " + dest)


def make_handler(fn):
    def inner(_):
        with log:
            clear_output()
            try:
                fn()
            except Exception as e:
                print("❌ Error: %s" % e)
    return inner


def run_all():
    process_auto("/transcribe_only", "transcription")
    process_auto("/translate_only", "translation")
    process_auto("/dub_only", "dub")


upload_btn.on_click(on_upload)
btn_transcribe.on_click(make_handler(lambda: process_auto("/transcribe_only", "transcription")))
btn_translate.on_click(make_handler(lambda: process_auto("/translate_only", "translation")))
btn_dub.on_click(make_handler(lambda: process_auto("/dub_only", "dub")))
btn_all.on_click(make_handler(run_all))

display(HTML("<b>یک ردیف برای همه حالت‌ها. قبل از هر دکمه، Upload را بزن، بعد ۱ → ۲ → ۳ یا Run all.</b>"))
display(widgets.HBox([upload_btn]))
display(widgets.HBox([btn_transcribe, btn_translate, btn_dub, btn_all]))
display(log)
